# AICE 핵심 코드 치트시트 (Cheat Sheet)

> 오픈북 시험에서 막힐 때 바로바로 찾아 복사/붙여넣기 할 수 있는 필수 코드 모음집입니다.

## 1. 조회·필터링 (loc / iloc)
```python
# loc: 라벨(컬럼명)·조건(Boolean) 기반 조회
df.loc[df['Age'] >= 30]                      # 조건에 맞는 행만
df.loc[df['Age'] >= 30, ['Age', 'Fare']]     # 조건 + 특정 컬럼만

# iloc: 위치(정수 인덱스) 기반 조회 - 이름이 아니라 몇 번째인지로 접근
df.iloc[0]         # 첫 번째 행
df.iloc[0:5, 0:3]  # 앞 5행, 앞 3컬럼

# 다중조건: &(AND, 둘 다 만족) / |(OR, 하나만 만족해도 됨) - 각 조건은 반드시 괄호로 감싸기
df.loc[(df['Age'] >= 30) & (df['Fare'] > 50)]
df.loc[(df['Pclass'] == 1) | (df['Pclass'] == 2)]
```

## 2. 결측치 처리
```python
# 결측치 확인
print(df.isnull().sum())

# 수치형 변수 중앙값으로 대체
df['Age'] = df['Age'].fillna(df['Age'].median())

# 범주형 변수 최빈값으로 대체 (최빈값은 Series 반환이라 [0] 인덱싱 필요)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
```

## 3. 데이터 전처리 (인코딩 & 분할 & 스케일링)
```python
# 원-핫 인코딩 (다중공선성 방지를 위해 drop_first=True)
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Train-Test 분할 (분류 모델일 경우 stratify=y 필수)
from sklearn.model_selection import train_test_split
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 스케일링 (학습 데이터는 fit_transform, 테스트 데이터는 transform)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
```

## 4. 머신러닝 5단계 템플릿
```python
# [1] 모델 불러오기
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# [2] 모델 생성
model = RandomForestClassifier(n_estimators=100, random_state=42)

# [3] 모델 학습
model.fit(X_train_s, y_train)

# [4] 예측
pred = model.predict(X_test_s)

# [5] 평가
print(accuracy_score(y_test, pred))
```

## 5. 딥러닝(Keras) 뼈대 코드
```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# 모델 구성
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dense(16, activation='relu'))
# 출력층: 회귀(1, linear), 이진분류(1, sigmoid), 다중분류(클래스개수, softmax)
model.add(Dense(1, activation='sigmoid'))

# 컴파일: 회귀(mse), 이진분류(binary_crossentropy), 다중분류(sparse_categorical_crossentropy)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 조기 종료
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 학습
history = model.fit(X_train_s, y_train, epochs=100, batch_size=32, validation_data=(X_test_s, y_test), callbacks=[es], verbose=1)
```



## 🚨 [핵심 기출] 무조건 알아야 하는 고득점 3대장 🚨

### 1. 고급 결측치 처리 (특정 그룹의 통계치로 대체)
> 단순 전체 평균이 아니라, 조건(예: 객실 등급별, 성별)에 따른 평균/중앙값으로 결측치를 정교하게 채우는 방법입니다.
```python
# Pclass(객실 등급)별 Age의 중앙값으로 결측치 채우기
df['Age'] = df['Age'].fillna(df.groupby('Pclass')['Age'].transform('median'))
```

### 2. 시계열(날짜/시간) 파생변수 생성
> 날짜 형태의 문자열이 주어졌을 때, 연, 월, 일, 요일 컬럼을 새롭게 만들어내는 전처리 단골 문제입니다.
```python
# 문자열을 datetime 객체로 변환
df['date_col'] = pd.to_datetime(df['date_col'])

# 파생변수 추출
df['year'] = df['date_col'].dt.year
df['month'] = df['date_col'].dt.month
df['day'] = df['date_col'].dt.day
df['dayofweek'] = df['date_col'].dt.dayofweek  # 0:월요일 ~ 6:일요일
```

### 3. 학습된 딥러닝 모델 저장 및 불러오기
> AICE 가장 마지막 문제로 항상 출제되는 모델 파일(.h5) 저장 및 로드 코드입니다.
```python
from tensorflow.keras.models import load_model

# 모델 저장하기
model.save('best_model.h5')

# 모델 불러오기
loaded_model = load_model('best_model.h5')

# 불러온 모델로 평가하기
loss, acc = loaded_model.evaluate(X_test_s, y_test)
print(f"Loaded Model Accuracy: {acc}")
```

